# *data cleaning*

## handeling missing values

In [86]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # Explicitly required
from sklearn.impute import IterativeImputer

In [87]:
df=pd.DataFrame({
    'age': [25, np.nan, 30, 45, 22,22],
    'income': [42430, 60000, np.nan, 80000, 45000,450000],
    'department': ['Tech', 'HR', 'Tech',np.nan, 'Marketing','Marketing']
})


imputation strategy

In [88]:
# simple imputer
mean_imputer = SimpleImputer(strategy="mean")
df["age"]=mean_imputer.fit_transform(df[["age"]])

median_imputer = SimpleImputer(strategy="median")
df["income"]=median_imputer.fit_transform(df[["income"]])

mode_imputer = SimpleImputer(strategy='most_frequent')
df['department'] = mode_imputer.fit_transform(df[['department']]).ravel()

In [89]:
# knnimputer
knn_imputer = KNNImputer(n_neighbors=2)
df[['age', 'income']] = knn_imputer.fit_transform(df[['age', 'income']])

In [90]:
df

,age,income,department
0,25.0,42430.0,Tech
1,28.8,60000.0,HR
2,30.0,60000.0,Tech
3,45.0,80000.0,Marketing
4,22.0,45000.0,Marketing
5,22.0,450000.0,Marketing


## remove duplicate

In [91]:
df = df.drop_duplicates(keep="first")
print(df)

    age    income department
0  25.0   42430.0       Tech
1  28.8   60000.0         HR
2  30.0   60000.0       Tech
3  45.0   80000.0  Marketing
4  22.0   45000.0  Marketing
5  22.0  450000.0  Marketing


In [92]:
df.duplicated().sum()

np.int64(0)

## outlier detection

In [93]:
q1=df['income'].quantile(0.25)
q3=df['income'].quantile(0.75)
iqr=q3-q1
upperbound=q3+1.5*iqr
lowerbound=q1-1.5*iqr

print(upperbound)
print(q1)
print(q3)
print(1.5*iqr)
print(lowerbound)
df['income_clipped'] = df['income'].clip(lower=lowerbound, upper=upperbound)
df

114375.0
48750.0
75000.0
39375.0
9375.0


,age,income,department,income_clipped
0,25.0,42430.0,Tech,42430.0
1,28.8,60000.0,HR,60000.0
2,30.0,60000.0,Tech,60000.0
3,45.0,80000.0,Marketing,80000.0
4,22.0,45000.0,Marketing,45000.0
5,22.0,450000.0,Marketing,114375.0


## datatype convertion


In [96]:
df.dtypes

age               float64
income            float64
department         object
income_clipped    float64
dtype: object

## df advanced operations

In [191]:
data = {
    "raw_price": ["$ 1,200.50 ", " $450.00 ", "$89.99"],
    "user_info": ["ID-901_JohnDoe", "ID-342_JaneSmith", "ID-115_AlexJones"],
    "raw_dates": ["2026/05/15", "2026-05-18 14:30:00", "May 23, 2026"],
    "age":["12","twelve",13]
}
df = pd.DataFrame(data)
print("--- Original Dataframe ---")
print(df, "\n")

--- Original Dataframe ---
     raw_price         user_info            raw_dates     age
0  $ 1,200.50     ID-901_JohnDoe           2026/05/15      12
1     $450.00   ID-342_JaneSmith  2026-05-18 14:30:00  twelve
2       $89.99  ID-115_AlexJones         May 23, 2026      13 



In [201]:
#df['age_corr'] = df['age'].astype(int,errors="ignore")
df['age_corr'] = pd.to_numeric(df['age'], errors='coerce').astype("Int64")
df

,raw_price,user_info,raw_dates,age,age_corr
0,"$ 1,200.50",ID-901_JohnDoe,2026/05/15,12,12
1,$450.00,ID-342_JaneSmith,2026-05-18 14:30:00,twelve,<NA>
2,$89.99,ID-115_AlexJones,"May 23, 2026",13,13


In [200]:
df.dtypes

raw_price    object
user_info    object
raw_dates    object
age          object
age_corr      Int64
dtype: object

In [175]:
# 2. String Manipulation and Parsing
# Clean the price string by removing spaces and the dollar sign
df["cleaned_price"] = df["raw_price"].str.strip().str.replace("$", "",regex=False)
df["cleaned_price"] = df["cleaned_price"].str.replace(",", "", regex=False)

In [176]:
df

,raw_price,user_info,raw_dates,cleaned_price
0,"$ 1,200.50",ID-901_JohnDoe,2026/05/15,1200.50
1,$450.00,ID-342_JaneSmith,2026-05-18 14:30:00,450.00
2,$89.99,ID-115_AlexJones,"May 23, 2026",89.99


In [177]:
df["username"] = df["user_info"].str.split("_").str[1]

In [178]:
df

,raw_price,user_info,raw_dates,cleaned_price,username
0,"$ 1,200.50",ID-901_JohnDoe,2026/05/15,1200.50,JohnDoe
1,$450.00,ID-342_JaneSmith,2026-05-18 14:30:00,450.00,JaneSmith
2,$89.99,ID-115_AlexJones,"May 23, 2026",89.99,AlexJones


In [179]:
df["float_price"] = df["cleaned_price"].astype(float)

In [180]:
df.dtypes

raw_price         object
user_info         object
raw_dates         object
cleaned_price     object
username          object
float_price      float64
dtype: object

In [187]:
df["datetime_obj"] = pd.to_datetime(df["raw_dates"].str.strip(), format="mixed" ,errors="coerce")

format="mixed" #helps to evaluate any kind of date format

In [188]:
df

,raw_price,user_info,raw_dates,cleaned_price,username,float_price,datetime_obj,formatted_date
0,"$ 1,200.50",ID-901_JohnDoe,2026/05/15,1200.50,JohnDoe,1200.50,2026-05-15 00:00:00,15/05/2026
1,$450.00,ID-342_JaneSmith,2026-05-18 14:30:00,450.00,JaneSmith,450.00,2026-05-18 14:30:00,NaN
2,$89.99,ID-115_AlexJones,"May 23, 2026",89.99,AlexJones,89.99,2026-05-23 00:00:00,NaN


In [189]:
df.dtypes

raw_price                 object
user_info                 object
raw_dates                 object
cleaned_price             object
username                  object
float_price              float64
datetime_obj      datetime64[ns]
formatted_date            object
dtype: object

In [190]:
df["formatted_date"] = df["datetime_obj"].dt.strftime("%d/%m/%Y")
df

,raw_price,user_info,raw_dates,cleaned_price,username,float_price,datetime_obj,formatted_date
0,"$ 1,200.50",ID-901_JohnDoe,2026/05/15,1200.50,JohnDoe,1200.50,2026-05-15 00:00:00,15/05/2026
1,$450.00,ID-342_JaneSmith,2026-05-18 14:30:00,450.00,JaneSmith,450.00,2026-05-18 14:30:00,18/05/2026
2,$89.99,ID-115_AlexJones,"May 23, 2026",89.99,AlexJones,89.99,2026-05-23 00:00:00,23/05/2026
